# Tutorial 14: End-to-End Coherent Scattering Experiment

This is the maintained replacement for the old monolithic scattering notebooks. It defines one complete magnetic Fourier-transform holography experiment, simulates matched CR and CL measurements, and compares charge-like sums with magnetic helicity differences at every stage.

The workflow is: illumination/source → experimental geometry → sample and mask → exit waves → ideal holograms → corrupted detector images → FTH reconstructions.


---

**This copy adds a chemical columnar-domain texture** in the CoDy region (Co-rich / Dy-rich nucleation-and-growth model, replacing the nominal deposited layering with the segregated structure seen by SAXS/EDX) alongside the original magnetic domain pattern. New/changed sections are marked `[CHEMICAL DOMAINS]`.

In [ ]:
# Sample-to-detector propagation (independent of multislice).
detector_propagation_method = "fraunhofer"  # Opt in with "rayleigh_sommerfeld".
# Direct Rayleigh-Sommerfeld is expensive: try small grids first.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
src_dir = repo_root / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim
plt.rcParams["figure.constrained_layout.use"] = True


# [CHEMICAL DOMAINS] extra imports for the columnar composition model
from scipy.ndimage import distance_transform_edt
from scattering_calculator.database.database_loading import material_params
from scattering_calculator.sample_generator.pattern_generator import (
    _estimate_labyrinth_stripe_width_fft,
)


## 1. Define the illumination and X-ray source

Define the photon energy, flux, CR/CL states, transverse coherence length, and spatial beam parameters first. The coherence length belongs conceptually to the illumination even though it is stored on `XRayConfig` and applied when the ideal far field is projected onto the detector. Larger coherence lengths preserve finer interference fringes.


In [ ]:
# ########################################
# ILLUMINATION AND X-RAY SOURCE
# ########################################
xray_energy = 778.0                       # eV
photon_flux = 5e8                         # photons / second
polarizations = ("CR", "CL")
transverse_coherence_length = (100e-6, 100e-6)  # m, (y, x)

illumination_function = "gaussian"
illumination_center = np.array([0.0, 0.0])       # m, (y, x)
illumination_focus_distance = 0.0                # m from beam waist
illumination_fwhm = 60.0e-6                      # m at beam waist
illumination_alpha_beam = (0.0, 0.0)             # rad, (alpha_y, alpha_x)

xray = sim.XRayConfig(
    energy=xray_energy, photon_flux=photon_flux, pol=polarizations[0],
    coherence_length=transverse_coherence_length,
)
xray.setup()


## 2. Define detector, acquisition, corruption, and beamstop parameters

The detector geometry and source wavelength determine the sample-plane pixel size. Acquisition parameters control the frame stack. Detector-response parameters convert photons to counts and add readout effects. Photon-event kernels are a separate corruption mechanism: `sigma_photon=0` disables this spreading entirely, while smaller positive sigma and kernel size values produce sharper events. Beamstop `sigma` controls only the physical beamstop-edge transition and is not the photon-kernel width.


**[CHEMICAL DOMAINS] Grid size note:** the CoDy columnar texture has a ~12 nm in-plane periodicity. `real_space_pixel_size` below is *derived* from `oversampling` and the detector geometry (`wavelength * sample_to_detector_distance / (detector_shape * detector_pixel_size) / oversampling`), not set directly. `oversampling=2` (tutorial default) gives only ~2 px per 12 nm period -- too coarse. Use `oversampling=8` (~8 px/period, comfortable) for a real run; `oversampling=4` (~4 px/period, borderline) is a cheaper sanity-check grid. This is the main lever for 'running on a larger grid'.

In [ ]:
# ########################################
# DETECTOR GEOMETRY
# ########################################
detector_shape = (512, 512)
detector_pixel_size = 20e-6       # m
sample_detector_distance = 0.075  # m
detector_center = tuple(np.array(detector_shape) // 2)
oversampling = 4  # [CHEMICAL DOMAINS] was 2; need >=4 to resolve the 12 nm periodicity. Try 8 on GPU for a sharper cross-section.

# ########################################
# ACQUISITION
# ########################################
number_frames = 100
max_counts_per_frame = 64e3
exposure_time = 1.0               # s per frame

# ########################################
# DETECTOR RESPONSE
# ########################################
readout_noise_average = 20
readout_noise_sigma = 3
detector_threshold = 64e3
counts_per_photon = 180
quantum_efficiency = 0.9
detector_noise_seed = 12

# ########################################
# PHOTON-EVENT KERNELS
# ########################################
sigma_photon = 0.15                # pixels; set 0.0 for no event spreading
photon_kernel_size = 9             # pixels; odd integer
photon_n_classes = 1
photon_n_variants = 30
photon_irregularity = 2.0
regenerate_photon_kernels = False
photon_kernel_seed = 21

# ########################################
# BEAMSTOP
# ########################################
beamstop_distance = 0.02          # m from detector
beamstop_radius = 0.12e-3            # m
beamstop_edge_sigma = 1e-6      # m; broad edge smoothing, independent of photon kernels
beamstop_wire_width = 10e-6   # m
beamstop_wire_bend = 15e-6         # m
beamstop_angle = np.deg2rad(20)

beamstop = sim.BeamstopConfig(
    bs_method="circular", bs_detector_distance=beamstop_distance,
    bs_center=detector_center,
    bs_config={
        "radius": beamstop_radius, "sigma": beamstop_edge_sigma,
        "wire_width": beamstop_wire_width, "wire_bend": beamstop_wire_bend,
        "angle": beamstop_angle, "antialias": 3, "seed": 4,
    },
)
detector = sim.DetectorConfig(
    detector_propagation_method=detector_propagation_method,
    shape=detector_shape, pixel_size=detector_pixel_size,
    sample_to_detector_distance=sample_detector_distance,
    detector_center=detector_center, beamstop_config=beamstop,
    detector_params={
        "readout_noise_average": readout_noise_average,
        "readout_noise_sigma": readout_noise_sigma,
        "detector_threshold": detector_threshold,
        "counts_per_photon": counts_per_photon,
        "quantum_efficiency": quantum_efficiency,
        "noise_seed": detector_noise_seed,
    },
    artifacts_config={
        "sigma_photon": sigma_photon,
        "photon_kernel_size": photon_kernel_size,
        "photon_n_classes": photon_n_classes,
        "photon_n_variants": photon_n_variants,
        "photon_irregularity": photon_irregularity,
        "regenerate_photon_kernels": regenerate_photon_kernels,
        "photon_kernel_seed": photon_kernel_seed,
    },
    measurement_config={
        "exposure_time": exposure_time,
        "number_frames": number_frames,
        # The API calls each frame an image, so this is the per-frame cap.
        "max_counts_per_image": max_counts_per_frame,
    },
)
detector.setup()
detector.visualize_beamstop()
real_space_pixel_size = detector.calc_realspace_resolution(xray.beam_params) / oversampling
sample_shape = [0, oversampling * detector_shape[0], oversampling * detector_shape[1]]
print(f"detector: {detector_shape}, distance: {sample_detector_distance:.3f} m")
print(f"acquisition: {number_frames} frames, maximum {max_counts_per_frame:,} counts per frame")
print(f"sample grid: {sample_shape[1:]}, pixel: {real_space_pixel_size * 1e9:.2f} nm")
print(f"coherence: {np.asarray(transverse_coherence_length) * 1e6} um")
print(f"photon kernel: sigma={sigma_photon} px, size={photon_kernel_size} px")


## 3. Define the sample geometry

**[CHEMICAL DOMAINS]** The real deposited recipe is `Ta(4)/Pt(6)/[Co(0.75)/Dy(0.3)]x120/Ta(3)/Pt(4)` (126 nm nominal CoDy stack). At 0.75 nm Co / 0.3 nm Dy per bilayer the film has almost certainly interdiffused into the columnar Co-rich/Dy-rich texture seen in SAXS/EDX (~12 nm in-plane periodicity) rather than remaining as discrete monolayers, so we do not encode the nominal bilayers directly (240 sub-nm z-slices would be both wrong and hugely expensive). Instead the CoDy region is subdivided into `N_SUB` coarser depth slices (same 126 nm total thickness, same Ta/Pt capping layers as the real stack) used only to resolve the columnar cone growth with depth; each slice's composition is set explicitly below from a nucleation-and-growth model, not from the recipe string.

The magnetic domain pattern (from the original tutorial) is kept as-is and assigned on top of this -- a domain wall sitting in the field of view is fine.

**[CHEMICAL DOMAINS] Simplified from the per-domain-random version**: single global `start_layer` (no per-domain nucleation spread, no `scipy.ndimage.label` bookkeeping). Fewer parameters, same underlying growth mechanism (distance-to-domain-wall threshold, growing outward from each domain's interior as depth increases).

**Important resolution note, found by testing this at the real target pixel size:** a 12 nm domain is only ~4 px wide at ~2.9 nm/px (`oversampling=4`). That leaves only 2-3 distinguishable radius steps between a domain's core and its wall, so at the real simulation resolution growth genuinely completes in just 2-3 sublayers -- not a bug, a real consequence of the periodicity being close to the resolution limit. The preview cell right after this one uses a finer pixel size *just for visualization* so you can see the underlying mechanism produce a properly gradual cone; the real run below it uses your actual physical resolution and will look like a sharper transition, which is physically correct, not wrong.

In [ ]:
# ########################################
# [CHEMICAL DOMAINS] columnar domain builder (simplified: one global start_layer)
# ########################################
def build_columnar_domains(base_pattern_2d, n_sub, start_layer, widen_rate_px_per_layer,
                            richness, f_background, max_radius_px):
    """Return (n_sub, Ny, Nx) Co-fraction field f in [0, 1].

    All domains start growing together at `start_layer`, expanding outward from
    each domain's own interior point as depth (sublayer index) increases, and
    stop once they reach `max_radius_px` (their natural periodicity-sized
    footprint). Below `start_layer` and before a pixel's domain has grown out
    to it, that pixel reads as the bulk alloy background `f_background`
    (not yet segregated). `richness` in [0,1]: 1 = full binary Co/Dy contrast
    once grown, 0 = no compositional contrast anywhere.
    """
    binary = base_pattern_2d > 0
    dist_pos = distance_transform_edt(binary)
    dist_neg = distance_transform_edt(~binary)
    dist_to_wall = np.where(binary, dist_pos, dist_neg)  # 0 at wall, higher toward domain interior
    pure = np.where(binary, 1.0, 0.0)

    f_stack = np.zeros((n_sub, *base_pattern_2d.shape))
    for z in range(n_sub):
        grown_radius = np.clip(widen_rate_px_per_layer * (z - start_layer), 0, max_radius_px)
        occupied = (max_radius_px - dist_to_wall) <= grown_radius
        raw = np.where(occupied, pure, f_background)
        f_stack[z] = f_background + richness * (raw - f_background)
    return f_stack


### 3a. [CHEMICAL DOMAINS] Quick preview -- tune parameters on a small, fast patch first

This uses a small 256x256 patch at a deliberately finer pixel size than the real run, purely so the growth cone is wide enough in pixels to look smooth while you're choosing `start_layer`/`widen_rate_px_per_layer`. It regenerates in seconds, unlike the full-size run below. Once you're happy with the shape here, section 3c re-checks the same parameters at your real simulation resolution.

In [ ]:
PREVIEW_SIZE = 256
PREVIEW_PIXEL_SIZE_NM = 0.75          # finer than the real run, just so growth is visible over several layers
PREVIEW_N_SUB = 15

preview_stripe_width_px = (12.0 / 2) / PREVIEW_PIXEL_SIZE_NM
preview_seed, _ = pattern_generator.create_binary_labyrinth_pattern(
    sz_array=[PREVIEW_SIZE, PREVIEW_SIZE], stripe_width=preview_stripe_width_px,
    sigma=0.5, H=64, W=64, n_steps=60, region="custom", use_gpu=False, seed=11,
    k0=1.05, eps=0.8, target_mean=0.0, noise_amp=0.0,
    max_hole_area=4, saturation_fraction_threshold=0.01,
)

# these are the parameters to iterate on -- copy your chosen values into section 3c below
PREVIEW_START_LAYER = 3
PREVIEW_WIDEN_RATE_PX = preview_stripe_width_px / 8.0   # spreads growth over ~8 sublayers

f_preview = build_columnar_domains(
    preview_seed, PREVIEW_N_SUB, start_layer=PREVIEW_START_LAYER,
    widen_rate_px_per_layer=PREVIEW_WIDEN_RATE_PX, richness=1.0,
    f_background=0.7143, max_radius_px=preview_stripe_width_px,
)

y0 = PREVIEW_SIZE // 2
cross = f_preview[:, y0, :]
zoom_px = int(preview_stripe_width_px * 10)  # ~5 domain periods, wide enough to see a few cones
x0 = PREVIEW_SIZE // 2 - zoom_px // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].imshow(cross, cmap="RdBu_r", vmin=0, vmax=1, aspect="auto", origin="lower")
axes[0].set_title("preview: full width")
axes[0].set_xlabel("x (px)"); axes[0].set_ylabel("sublayer (depth)")
axes[1].imshow(cross[:, x0:x0 + zoom_px], cmap="RdBu_r", vmin=0, vmax=1, aspect="auto", origin="lower")
axes[1].set_title(f"preview: zoomed to ~{zoom_px} px (~5 periods) -- cone shape visible here")
axes[1].set_xlabel("x (px)"); axes[1].set_ylabel("sublayer (depth)")


In [ ]:
# ########################################
# [CHEMICAL DOMAINS] MATERIAL STACK
# real recipe: Ta(4)/Pt(6)/[Co(0.75)/Dy(0.3)]x120/Ta(3)/Pt(4) -> 126 nm CoDy stack.
# CoDy region kept at its real total thickness and real Ta/Pt caps, but subdivided
# into N_SUB coarser slices (not the literal 0.75/0.3 nm bilayers -- see markdown above).
# ########################################
CO_NOMINAL_NM, DY_NOMINAL_NM = 0.75, 0.3
CODY_TOTAL_NM = 120 * (CO_NOMINAL_NM + DY_NOMINAL_NM)   # 126.0 nm
f_background = CO_NOMINAL_NM / (CO_NOMINAL_NM + DY_NOMINAL_NM)  # 0.7143, bulk stoichiometric ratio

N_SUB = 15                          # depth slices for the cone model; raise for a finer cross-section
SUB_THICKNESS_NM = CODY_TOTAL_NM / N_SUB

# ########################################
# Real physical stack, beam-first order: Au mask (opaque, holes drilled through
# it) -> SiN membrane -> Ta/Pt/CoDy/Ta/Pt sample film. The mask is a physically
# separate, much thicker layer from the sample itself -- earlier versions of
# this notebook omitted it entirely and drilled holes directly into the thin
# sample film, which was wrong (see markdown above this cell).
# ########################################
AU_MASK_THICKNESS_NM = 4000.0
SIN_THICKNESS_NM = 50.0

sample = sim.SampleConfig(
    recipe=(
        f"Au({AU_MASK_THICKNESS_NM})/SiN({SIN_THICKNESS_NM})/"
        f"Ta(4)/Pt(6)/[Co({SUB_THICKNESS_NM})]x{N_SUB}/Ta(3)/Pt(4)"
    ),
    sample_shape=sample_shape, real_space_pixel_size=real_space_pixel_size,
    xray_config=xray, sample_name="chemical columnar-cone tutorial",
)
sample.setup()
n_layers = len(sample.sample_structure.layer_names)
chem_indices = [i for i, n in enumerate(sample.sample_structure.layer_names) if n == "Co"]
sample.sample_structure.visualize_structure()
print("layers:", sample.sample_structure.layer_names)
print("n layers:", n_layers, " CoDy sublayers:", len(chem_indices), f"({SUB_THICKNESS_NM:.2f} nm each)")

mp = material_params(materials={"Co", "Dy"}, x_ray_energy=xray.energy)
n_co = np.asarray(mp.get_refractive_index("Co"), dtype=complex)
n_dy = np.asarray(mp.get_refractive_index("Dy"), dtype=complex)

## ########################################
# [CHEMICAL DOMAINS] isotropic ~12 nm-period seed pattern (SAXS/EDX-matched, bicontinuous)
# "stripe_width" convention ~= half the FFT-measured full period.
#
# PERFORMANCE NOTE: this is an FFT-based spectral generator, not a literal
# stepped PDE solver, so it runs fine on CPU (no CUDA/CuPy required) -- but at
# large grids it has to run near full sample resolution (can't shrink much
# below the target periodicity), and create_binary_labyrinth_pattern retries
# internally 2-6x while converging on the right base size. Rough CPU timings
# (single call): 1024^2 ~8s, 2048^2 ~40s -- so oversampling=4 may take a few
# minutes total including retries, oversampling=8 several minutes. It is a
# one-time cost per grid size thanks to the disk cache below. SEED_USE_GPU is
# a no-op without cupy installed (falls back to numpy automatically) -- safe
# to leave False on a Mac.
#
# The cache filename includes every generator parameter that affects the
# result (period, k0, eps, seed) -- if you change any of those you get a
# fresh cache entry rather than a silently stale one.
# ########################################
SEED_USE_GPU = False
target_period = 12e-9
k0_val, eps_val, seed_val = 1.05, 0.8, 11

def _generate_seed_pattern(stripe_width_px):
    return pattern_generator.create_binary_labyrinth_pattern(
        sz_array=list(sample_shape[1:]), stripe_width=stripe_width_px,
        sigma=0.5, H=64, W=64, n_steps=60, region="custom", use_gpu=SEED_USE_GPU,
        seed=seed_val, k0=k0_val, eps=eps_val, target_mean=0.0, noise_amp=0.0,
        max_hole_area=4, saturation_fraction_threshold=0.01,
        max_auto_size=max(4096, 2 * max(sample_shape[1:])),
        max_auto_pixels=max(4096, 2 * max(sample_shape[1:])) ** 2,
    )

def _measured_period_nm(pattern):
    """Measure periodicity directly on the delivered output array, in the
    real sample pixel grid. `seed_meta["measured_period_px"]` from the
    library is NOT usable for this -- it is measured on the generator's
    internal pre-rescale working grid, which has a different (and not
    directly recoverable from the returned dict) pixel pitch."""
    _, period_px = _estimate_labyrinth_stripe_width_fft(pattern)
    return period_px * real_space_pixel_size * 1e9

seed_cache_path = Path(
    f"seed_pattern_cache_{sample_shape[1]}x{sample_shape[2]}"
    f"_period{target_period*1e9:.1f}nm_k0-{k0_val}_eps-{eps_val}_seed{seed_val}.npz"
)
stripe_width_px = (target_period / 2) / real_space_pixel_size  # also reused below for max_radius_px
if seed_cache_path.exists():
    _cached = np.load(seed_cache_path, allow_pickle=True)
    seed_pattern, seed_meta = _cached["pattern"], _cached["meta"].item()
    print("loaded cached seed pattern from", seed_cache_path)
else:
    seed_pattern, seed_meta = _generate_seed_pattern(stripe_width_px)
    measured_nm = _measured_period_nm(seed_pattern)
    rel_err = abs(measured_nm - target_period * 1e9) / (target_period * 1e9)
    print(f"first pass measured period (on delivered pattern): {measured_nm:.2f} nm "
          f"(target {target_period*1e9:.1f} nm, {rel_err*100:.0f}% off)")
    if rel_err > 0.15:  # occasional first-pass miss; correct once
        stripe_width_px_corrected = stripe_width_px * (target_period * 1e9) / measured_nm
        if stripe_width_px_corrected < 1.0:
            print(f"WARNING: corrected stripe_width_px={stripe_width_px_corrected:.3f} is sub-pixel -- "
                  "real_space_pixel_size is too coarse to resolve the target period at all. "
                  "Skipping the correction pass; increase oversampling instead of relying on this.")
        else:
            print(f"re-generating with corrected stripe_width_px={stripe_width_px_corrected:.2f} "
                  f"(was {stripe_width_px:.2f})")
            seed_pattern, seed_meta = _generate_seed_pattern(stripe_width_px_corrected)
            measured_nm = _measured_period_nm(seed_pattern)
            print(f"corrected measured period: {measured_nm:.2f} nm")
    np.savez(seed_cache_path, pattern=seed_pattern, meta=seed_meta)
print("final measured period (on delivered pattern):", _measured_period_nm(seed_pattern),
      "nm  (target 12 nm)")

# ########################################
# [CHEMICAL DOMAINS] columnar growth -> Co-fraction stack
# same start_layer/widen_rate you tuned on the small preview above (section 3a);
# max_radius_px reuses stripe_width_px since that's each domain's natural half-period
# ########################################
f_stack = build_columnar_domains(
    seed_pattern, N_SUB,
    start_layer=3,                              # copy from your preview tuning
    widen_rate_px_per_layer=stripe_width_px / 3, # copy from your preview tuning
    richness=1.0,                               # 1=full binary Co/Dy; soften later e.g. 0.6
    f_background=f_background,
    max_radius_px=stripe_width_px,
)
f_stack_top_only = np.repeat(f_stack[-1:, :, :], N_SUB, axis=0)  # comparison baseline, see section 9 below

# ########################################
# MAGNETIC PATTERN -- unchanged from the original tutorial. A domain wall sitting
# in the field of view is fine; this is independent of the chemical texture above.
# ########################################
magnetic = sim.MagneticPatternConfig(
    pattern_type_method="binary_labyrinth_pattern",
    shape=tuple(sample_shape[1:]), real_space_pixel_size=real_space_pixel_size,
    pattern_config={
        "stripe_width": 300e-9, "sigma": 5e-9,
        "H": 128, "W": 128, "n_steps": 60,
        "region": "custom", "use_gpu": False, "seed": 7,
        "k0": 1.05, "eps": 0.8, "target_mean": 0.0,
        "noise_amp": 0.0, "quadratic_coefficient": 0.0,
        "max_hole_area": 9, "saturation_fraction_threshold": 0.01,
    },
)
magnetic.create_pattern()
mz = magnetic.magnetic_pattern
magnetization = pattern_generator.map_magnetization_to_3d(
    magnetic_pattern_x=np.zeros_like(mz),
    magnetic_pattern_y=np.sqrt(np.clip(1.0 - mz**2, 0.0, 1.0)),
    magnetic_pattern_z=mz,
    nr_repeats=n_layers,
)
sample.assign_magnetic_pattern(magnetization)


### 3b. [CHEMICAL DOMAINS] Verify the columnar cone geometry

`visualize_structure()` and the printed layer list above only reflect the placeholder recipe (all sublayers typed `Co`) -- they don't show the actual Co/Dy mixing, which is only applied later via `f_stack`. This is the real check: top-down views at a few depths, plus a side (depth) profile through the film so the nucleation-and-growth geometry can be inspected directly before it goes into the dielectric tensor.

In [ ]:
# top-down views at a few representative depths
extent_um = np.array([-sample_shape[2]/2, sample_shape[2]/2, sample_shape[1]/2, -sample_shape[1]/2]) * real_space_pixel_size * 1e6
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
show_layers = sorted(set([0, N_SUB // 3, 2 * N_SUB // 3, N_SUB - 1]))
for ax, z in zip(axes, show_layers):
    tag = " (bottom/substrate)" if z == 0 else " (top/surface)" if z == N_SUB - 1 else ""
    ax.imshow(f_stack[z], cmap="RdBu_r", vmin=0, vmax=1, extent=extent_um)
    ax.set_title(f"layer {z}/{N_SUB-1}{tag}")
    ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")
plt.suptitle("Co fraction f(x,y) at selected depths  (1=Co-rich, 0=Dy-rich, "
             f"{f_background:.2f}=unsegregated bulk alloy)")

# side profile: depth cross-section at fixed y, substrate at the bottom of the
# plot and film surface at the top, matching physical intuition. Two panels:
# full width (for context) and zoomed to a handful of periods (to actually see
# the cone taper -- at full sample width each domain is only a few px wide).
y0 = f_stack.shape[1] // 2
cross = f_stack[:, y0, :]  # (N_SUB, Nx)
zoom_px = int(stripe_width_px * 10)  # ~5 domain periods
x0 = f_stack.shape[2] // 2 - zoom_px // 2

extent_cross_full = [
    -sample_shape[2] / 2 * real_space_pixel_size * 1e6,
    sample_shape[2] / 2 * real_space_pixel_size * 1e6,
    0, CODY_TOTAL_NM,
]
extent_cross_zoom = [
    -zoom_px / 2 * real_space_pixel_size * 1e9,
    zoom_px / 2 * real_space_pixel_size * 1e9,
    0, CODY_TOTAL_NM,
]
fig2, axes2 = plt.subplots(1, 2, figsize=(15, 4.5))
im0 = axes2[0].imshow(cross, cmap="RdBu_r", vmin=0, vmax=1, aspect="auto",
                       extent=extent_cross_full, origin="lower")
axes2[0].set_title(f"side profile at y={y0*real_space_pixel_size*1e6:.2f} um -- full width")
axes2[0].set_xlabel("x (um)"); axes2[0].set_ylabel("depth into CoDy film (nm), 0 = substrate side")
im1 = axes2[1].imshow(cross[:, x0:x0 + zoom_px], cmap="RdBu_r", vmin=0, vmax=1, aspect="auto",
                       extent=extent_cross_zoom, origin="lower")
axes2[1].set_title(f"zoomed to ~{zoom_px} px (~5 periods) -- cone shape visible here")
axes2[1].set_xlabel("x (nm)"); axes2[1].set_ylabel("depth into CoDy film (nm)")
fig2.colorbar(im1, ax=axes2, label="Co fraction f", shrink=0.85)

print(f"fraction still unsegregated background (f~{f_background:.3f}) at bottom layer:",
      f"{np.mean(np.isclose(f_stack[0], f_background, atol=0.02)):.2%}")
print(f"fraction still unsegregated background (f~{f_background:.3f}) at top layer:",
      f"{np.mean(np.isclose(f_stack[-1], f_background, atol=0.02)):.2%}")


### 4. Holography mask (real geometry)

**Fixed a real bug here.** The previous version drilled OH/RH holes directly into the thin sample film -- there was no separate mask layer at all. Now the recipe (section 3) includes the real 4 um Au mask and 50 nm SiN membrane upstream of the sample, and the object hole is drilled through the mask + membrane only (`thickness_OH = Au + SiN`), stopping right at the sample surface so the sample film itself stays intact and is what actually gets imaged. Reference holes still drill through everything, matching the library's default RH behavior.

In [ ]:
# ########################################
# HOLOGRAPHY MASK: OBJECT AND REFERENCE HOLES
# ########################################
thicknesses = sample.sample_structure.layer_thicknesses
membrane_index = sample.sample_structure.layer_names.index("SiN")
thickness_OH = float(np.sum(thicknesses[:membrane_index + 1]))  # through Au mask + SiN, stops at the sample
print(f"thickness_OH = {thickness_OH*1e9:.1f} nm (Au mask + SiN membrane)")

aperture = sim.FrontApertureConfig(
    aperture_method="FTH_circular", aperture_shape=sample_shape,
    real_space_pixel_size=real_space_pixel_size,
    aperture_thicknesses=thicknesses,
    aperture_layer_names=sample.sample_structure.layer_names,
    aperture_config={
        "apertures_type": ["OH", "RH", "RH"],
        "apertures_radius": [0.5e-6, 60e-9, 30e-9],
        "apertures_center": [(0.0, 0.0), (1.1e-6, -1.1e-6), (-1.1e-6, -1.1e-6)],
        "apertures_sigma": [4e-9, 2e-9, 2e-9],
        "apertures_angle": [0.0, 0.0, 0.0],
        "apertures_ellipticity": [1.0, 1.0, 1.0],
        "apertures_roughness": [0.0, 0.0, 0.0],
        "apertures_roughness_modes": [(0, 0), (0, 0), (0, 0)],
        "apertures_seed": [1, 2, 3],
        "apertures_top_radius_factor": [1.0, 1.0, 1.0],
        "aperture_taper_depth": 0.0,
        "thickness_OH": thickness_OH,
    },
    use_roi=True,
)
aperture.setup()
aperture.visualize_aperture()
aperture_mask = aperture.return_aperture()
sample.assign_aperture_mask(aperture_mask)

mask_projection = np.mean(aperture_mask, axis=0)  # mean, not max -- OH only opens 2/14 layers (mask+membrane),
                                                     # so max would hide it entirely; mean shows it as partial (grey)
extent_um = np.array([-sample_shape[2]/2, sample_shape[2]/2, sample_shape[1]/2, -sample_shape[1]/2]) * real_space_pixel_size * 1e6
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
axes[0].imshow(mz, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um); axes[0].set_title("magnetic pattern")
axes[1].imshow(mask_projection, cmap="gray", extent=extent_um); axes[1].set_title("OH + RH mask")
axes[2].imshow(mz * mask_projection, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_um); axes[2].set_title("visible domains")
for ax in axes: ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")

# ########################################
# [CHEMICAL DOMAINS] compute the full dielectric tensor (mask + magnetism), then
# splice in the composition-mixed isotropic term for the CoDy sublayers, keeping
# whatever magnetic (mz/xy) contribution Structure already computed for them.
# NOTE: compact=False is required so final_dielectric_tensor is a dense array we
# can slice into directly.
# ########################################
def inject_chemical_composition(sample, chem_indices, f_stack, n_co, n_dy):
    """Overwrite the isotropic (composition) part of the CoDy sublayers' dielectric
    tensor in-place, preserving the magnetic (mz/xy) contribution already computed
    from the assigned magnetization pattern."""
    sample.sample_structure.calculate_final_dielectric_tensor(use_aperture_roi=True, compact=False)
    eps = sample.sample_structure.final_dielectric_tensor  # dense (Nz,Ny,Nx,2,2)
    mask = sample.sample_structure.mask
    m = sample.sample_structure.magnetization
    dt = np.asarray(sample.sample_structure.dielectric_tensors)  # (Nz,3,2,2): [eps0, eps_mz, eps_xy] per layer

    for rel, iz in enumerate(chem_indices):
        n0_local = f_stack[rel] * n_co[0] + (1.0 - f_stack[rel]) * n_dy[0]
        eps0_local = n0_local ** 2  # composition-mixed isotropic term, replaces the uniform-Co eps0

        eps_mz = dt[iz, 1]   # (2,2), Co's circular/XMCD tensor (constant across this sublayer)
        eps_xy = dt[iz, 2]   # (2,2), Co's linear/XMLD tensor
        mz_local = m[iz, ..., 2]
        mx_local, my_local = m[iz, ..., 0], m[iz, ..., 1]

        eps_local = np.zeros((*f_stack.shape[1:], 2, 2), dtype=eps.dtype)
        eps_local[..., 0, 0] = eps0_local
        eps_local[..., 1, 1] = eps0_local
        eps_local += mz_local[..., None, None] * eps_mz[None, None, :, :]
        eps_local += (np.abs(mx_local) ** 2 - np.abs(my_local) ** 2)[..., None, None] * eps_xy[None, None, :, :]

        mask_iz = mask[iz][..., None, None]
        eps[iz] = mask_iz * eps_local + (1.0 - mask_iz) * np.eye(2, dtype=eps.dtype)
    return eps

eps = inject_chemical_composition(sample, chem_indices, f_stack, n_co, n_dy)


## 4. Build the illumination on the resolved sample grid

The source and coherence parameters were chosen first. Now that detector geometry has fixed the sample grid, build the spatial illumination. `update_polarization` switches its Jones vector between the matched CR/CL helicities without rebuilding the scalar field.


In [ ]:
# ########################################
# SPATIAL ILLUMINATION ON THE SAMPLE GRID
# ########################################
illumination = sim.IlluminationConfig(
    XRayConfig=xray, shape=tuple(sample_shape[1:]),
    real_space_pixel_size=real_space_pixel_size,
    illumination_function=illumination_function,
    illumination_config={
        "center": illumination_center,
        "distance": illumination_focus_distance,
        "fwhm": illumination_fwhm,
        "alpha_beam": illumination_alpha_beam,
    },
)
illumination.setup()
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(np.abs(illumination.illumination.illumination)**2, cmap="magma", extent=extent_um)
ax.set_title("incident Gaussian intensity"); ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")


## 5. Simulate exit waves and detector holograms

For each helicity, Jones propagation produces the complex exit wave and the ideal far-field intensity. `DetectorConfig` then projects that intensity onto the physical detector and simulates the requested acquisition, applying finite coherence, the beamstop, photon statistics, photon-event kernels, readout noise, detector threshold, and the per-frame count ceiling. The current detector API returns the frame-averaged corrupted hologram; increasing `number_frames` reduces its statistical noise rather than returning a frame stack.


In [ ]:
# ########################################
# CR / CL PROPAGATION AND DETECTION
# ########################################
results = {}
for helicity in polarizations:
    illumination.update_polarization(helicity)
    propagation = sim.SamplePropagatorConfig(
        SampleConfig=sample, IlluminationConfig=illumination,
        propagator_method="Jones",
        propagator_config={
            "propagate": False, "jones_apply_zero_order_phase": True,
            "dielectric_tensor_use_roi": True,
        },
    )
    propagation.setup()
    detector.assign_propagated_wavefront(propagation)
    detector.detect_hologram()
    detected_average = detector.return_detected_hologram().copy()
    results[helicity] = {
        "exit": propagation.return_scalar_wavefield().copy(),
        "ideal": detector.return_ideal_hologram().copy(),
        "detected": detected_average,
    }
print("simulated:", list(results))
print("frame-averaged corrupted hologram shape:", results["CR"]["detected"].shape)
print("partial-coherence Gaussian sigma (y, x) in detector pixels:",
      (detector.hologram_exp.sigma_y, detector.hologram_exp.sigma_x))
print("photon-event spreading: sigma=", sigma_photon, "px; kernel size=", photon_kernel_size)


## 6. Inspect the CR and CL exit waves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for row, helicity in enumerate(("CR", "CL")):
    field = results[helicity]["exit"]
    amp = axes[row, 0].imshow(np.abs(field), cmap="magma", vmin=30, vmax=50, extent=extent_um)
    phase = axes[row, 1].imshow(np.angle(field), cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=extent_um)
    axes[row, 0].set_title(f"{helicity} exit amplitude")
    #axes[row, 0].set_xlim(-1, 1)
    #axes[row, 0].set_ylim(-1, 1)
    axes[row, 1].set_title(f"{helicity} exit phase")
    #axes[row, 1].set_xlim(-1, 1)
    #axes[row, 1].set_ylim(-1, 1)
    fig.colorbar(amp, ax=axes[row, 0], shrink=0.75); fig.colorbar(phase, ax=axes[row, 1], shrink=0.75)


## 7. Ideal and corrupted holograms: sum and difference

The sum `CR + CL` emphasizes nonmagnetic/charge scattering. The difference `CR - CL` isolates helicity-dependent magnetic contrast. The same algebra is applied to the ideal and corrupted detector images.


In [ ]:
holograms = {}
for source in ("ideal", "detected"):
    cr, cl = results["CR"][source], results["CL"][source]
    holograms[source] = {"CR": cr, "CL": cl, "sum": cr + cl, "difference": cr - cl}

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, source in enumerate(("ideal", "detected")):
    for col, channel in enumerate(("CR", "CL", "sum", "difference")):
        data = holograms[source][channel]
        if channel == "difference":
            limit = max(np.max(np.abs(data)), np.finfo(float).eps)
            image = axes[row, col].imshow(data, cmap="RdBu_r", vmin=-limit, vmax=limit)
        else:
            floor = max(np.max(data) * 1e-8, np.finfo(float).tiny)
            image = axes[row, col].imshow(np.log10(np.maximum(data, floor)), cmap="magma")
        axes[row, col].set_title(f"{source} {channel}"); axes[row, col].set_axis_off()
        fig.colorbar(image, ax=axes[row, col], shrink=0.72)


## 8. Reconstruct sums and differences

An FTH reconstruction is the centered Fourier transform of the detector hologram. Reconstructions contain displaced object images around each reference-hole correlation peak. Use the same display scale within each channel to compare ideal and corrupted data fairly.


In [ ]:
def fth_reconstruct(hologram):
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(hologram)))

reconstructions = {
    source: {channel: fth_reconstruct(holograms[source][channel]) for channel in ("sum", "difference")}
    for source in ("ideal", "detected")
}
fig, axes = plt.subplots(2, 2, figsize=(9, 8))
for row, source in enumerate(("ideal", "detected")):
    for col, channel in enumerate(("sum", "difference")):
        magnitude = np.abs(reconstructions[source][channel])
        vmax = np.percentile(magnitude, 90.)
        image = axes[row, col].imshow(magnitude, cmap="inferno", vmin=0, vmax=vmax)
        axes[row, col].set_title(f"{source} |FTH({channel})|"); axes[row, col].set_axis_off()
        fig.colorbar(image, ax=axes[row, col], shrink=0.75)


## Interpretation

- CR and CL share the structural scattering but interact oppositely with out-of-plane magnetization.
- Their sum is dominated by charge/structural contrast; their difference emphasizes magnetic circular contrast.
- The printed partial-coherence sigma distinguishes coherence blur from photon-event spreading. With a large coherence length it should approach zero pixels.
- Set `sigma_photon=0.0` to test the detector without photon kernels; `photon_kernel_size` is only the support window, while `sigma_photon` primarily controls event width.
- Beamstop `sigma` smooths the beamstop edge in physical detector coordinates and can also create a broad transition when chosen very large.
- The beamstop, finite photon statistics, detector response, and readout noise make the detected holograms more realistic and propagate into their reconstructions.
- Keep matched acquisition conditions for CR and CL; otherwise the difference also contains exposure or normalization mismatch.


## 9. [CHEMICAL DOMAINS] Nucleated columnar growth vs. a fully-segregated film

**Fixed a MemoryError here.** The original version rebuilt the *full* `(Nz, Ny, Nx, 2, 2)` complex128 dielectric tensor twice more (once for the top-only variant, once to restore the nucleated one) on top of everything already resident from earlier cells -- several GB each time at a 2048x2048 grid, which is how it ran out of memory.

The actual comparison only needs a small window around the object hole anyway (that's the request below: a line scan through it). Since `propagate=False` means the exit wave at each pixel only depends on the dielectric stack *directly under that pixel* (no lateral coupling), cropping the mask/magnetization/composition down to a small window *before* building the tensor is exact, not an approximation -- and it shrinks the computation from gigabytes to megabytes. The full nucleated `eps` computed back in section 4 is never touched here, so nothing needs restoring afterward.

In [ ]:
CROP_HALF_WIDTH_NM = 750.0  # window around the OH (radius 500 nm) -- generous margin
crop_half_px = int(CROP_HALF_WIDTH_NM * 1e-9 / real_space_pixel_size)
cy, cx = sample_shape[1] // 2, sample_shape[2] // 2  # OH is at (0,0), i.e. the array center
y0, y1 = cy - crop_half_px, cy + crop_half_px
x0, x1 = cx - crop_half_px, cx + crop_half_px
crop_extent_nm = [-crop_half_px * real_space_pixel_size * 1e9, crop_half_px * real_space_pixel_size * 1e9] * 2

mask_full = sample.sample_structure.mask
magnetization_full = sample.sample_structure.magnetization
mask_crop = mask_full[:, y0:y1, x0:x1]
magnetization_crop = magnetization_full[:, y0:y1, x0:x1, :]
f_stack_crop = f_stack[:, y0:y1, x0:x1]
f_stack_top_only_crop = f_stack_top_only[:, y0:y1, x0:x1]
crop_shape = (y1 - y0, x1 - x0)
print(f"crop shape: {crop_shape} px  (vs full {sample_shape[1:]}) -- "
      f"{(crop_shape[0]*crop_shape[1]) / (sample_shape[1]*sample_shape[2]):.4%} of the full array")

illumination_crop = sim.IlluminationConfig(
    XRayConfig=xray, shape=crop_shape, real_space_pixel_size=real_space_pixel_size,
    illumination_function="gaussian",
    illumination_config={"center": np.array([0.0, 0.0]), "distance": 0.0,
                          "fwhm": 400e-9, "alpha_beam": (0.0, 0.0)},
)
illumination_crop.setup()
illumination_crop.update_polarization("CR")

def exit_wave_for_crop(f_stack_variant):
    sample.sample_structure.mask = mask_crop
    sample.sample_structure.magnetization = magnetization_crop
    inject_chemical_composition(sample, chem_indices, f_stack_variant, n_co, n_dy)
    propagation_crop = sim.SamplePropagatorConfig(
        SampleConfig=sample, IlluminationConfig=illumination_crop,
        propagator_method="Jones",
        propagator_config={"propagate": False, "jones_apply_zero_order_phase": True,
                            "dielectric_tensor_use_roi": False},  # tiny crop -- ROI bookkeeping isn't worth it here
    )
    propagation_crop.setup()
    return propagation_crop.return_scalar_wavefield().copy()

exit_nucleated_crop = exit_wave_for_crop(f_stack_crop)
exit_top_only_crop = exit_wave_for_crop(f_stack_top_only_crop)

# restore the full-size arrays -- cheap, these were never freed, just swapped out
sample.sample_structure.mask = mask_full
sample.sample_structure.magnetization = magnetization_full


### Line scan through the object hole

The comparison that actually matters: a horizontal line through the OH center, amplitude and phase, nucleated vs. top-only overlaid on the same axes.

In [ ]:
y_line = exit_nucleated_crop.shape[0] // 2  # OH center row
x_axis_nm = np.linspace(crop_extent_nm[0], crop_extent_nm[1], exit_nucleated_crop.shape[1])

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].imshow(np.abs(exit_nucleated_crop), extent=crop_extent_nm, cmap="magma")
axes[0, 0].axhline(0, color="cyan", lw=1)
axes[0, 0].set_title("nucleated: |exit wave|")
axes[0, 1].imshow(np.abs(exit_top_only_crop), extent=crop_extent_nm, cmap="magma")
axes[0, 1].axhline(0, color="cyan", lw=1)
axes[0, 1].set_title("top-only (fully segregated): |exit wave|")
for ax in axes[0]: ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")

axes[1, 0].plot(x_axis_nm, np.abs(exit_nucleated_crop[y_line]), label="nucleated")
axes[1, 0].plot(x_axis_nm, np.abs(exit_top_only_crop[y_line]), label="top-only", alpha=0.8)
axes[1, 0].set_xlim(-100, 100)
axes[1, 0].set_ylim(5000, 8500)
axes[1, 0].set_title("line scan: amplitude")
axes[1, 0].set_xlabel("x (nm)"); axes[1, 0].set_ylabel("|exit wave|")
axes[1, 0].legend()

axes[1, 1].plot(x_axis_nm, np.angle(exit_nucleated_crop[y_line]), label="nucleated")
axes[1, 1].plot(x_axis_nm, np.angle(exit_top_only_crop[y_line]), label="top-only", alpha=0.8)
axes[1, 1].set_xlim(-100, 100)
axes[1, 1].set_ylim(1, 3)
axes[1, 1].set_title("line scan: phase")
axes[1, 1].set_xlabel("x (nm)"); axes[1, 1].set_ylabel("phase (rad)")
axes[1, 1].legend()


In [ ]:
def exit_wave_for_thin_film(n_layers_kept):
    """Only the top n_layers_kept CoDy sublayers are physically present (fully
    segregated); the rest are removed (mask=0, vacuum) rather than filled with
    anything -- tests thickness alone, no background padding."""
    n_removed = N_SUB - n_layers_kept
    removed_layer_indices = chem_indices[:n_removed]  # bottom-most sublayers, by the z convention used throughout

    mask_thin = mask_crop.copy()
    mask_thin[removed_layer_indices] = 0.0  # vacuum -- material genuinely absent, not unmixed background

    composition_full_res = np.repeat(f_stack[-1:], N_SUB, axis=0)[:, y0:y1, x0:x1]  # fully segregated wherever present

    sample.sample_structure.mask = mask_thin
    sample.sample_structure.magnetization = magnetization_crop
    inject_chemical_composition(sample, chem_indices, composition_full_res, n_co, n_dy)
    propagation_thin = sim.SamplePropagatorConfig(
        SampleConfig=sample, IlluminationConfig=illumination_crop,
        propagator_method="Jones",
        propagator_config={"propagate": False, "jones_apply_zero_order_phase": True,
                            "dielectric_tensor_use_roi": False},
    )
    propagation_thin.setup()
    return propagation_thin.return_scalar_wavefield().copy()

exit_waves_thin = {N_SUB: exit_top_only_crop}  # full-thickness, fully-segregated, already computed above
for n_kept in [5, 10]:
    exit_waves_thin[n_kept] = exit_wave_for_thin_film(n_kept)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for n_kept in sorted(exit_waves_thin):
    axes[0].plot(x_axis_nm, np.abs(exit_waves_thin[n_kept][y_line]), label=f"{n_kept}/{N_SUB} layers present")
axes[0].set_title("line scan: amplitude vs. film thickness\n(fully segregated throughout, no background)")
axes[0].set_xlim(-100, 100)
axes[0].set_ylim(4000, 12000)
axes[0].set_xlabel("x (nm)"); axes[0].set_ylabel("|exit wave|")
axes[0].legend()

for n_kept in sorted(exit_waves_thin):
    axes[1].plot(x_axis_nm, np.angle(exit_waves_thin[n_kept][y_line]), label=f"{n_kept}/{N_SUB} layers present")
axes[1].set_title("line scan: phase vs. film thickness")
axes[1].set_xlim(-100, 100)
axes[1].set_ylim(1, 3)
axes[1].set_xlabel("x (nm)"); axes[1].set_ylabel("phase (rad)")
axes[1].legend()

# restore full-thickness mask for anything run after this cell
sample.sample_structure.mask = mask_full